--- HIERARCHY ---
1. Document      (dossier, document)
    - mapping document<>dataset
2. Dataset       (report, OLAP cube, data import, internal dossier dataset)
3. Report Object (attribute, metric)
4. Schema Object (attribute, fact)
5. Table
6. Datasource


This script produces several CSV files. Of course, you can push this data directly to cubes.

In [20]:
import csv
import pandas as pd
import getpass
from mstrio.connection import Connection
from mstrio.server import Environment, Project
from mstrio.object_management import full_search
from mstrio.types import ObjectSubTypes, ObjectTypes

from mstrio import config
config.verbose = False

In [2]:
mstr_username = "mstr"
mstr_password = getpass.getpass(prompt='Password ')
mstr_base_url = "https://env-306570.customer.cloud.microstrategy.com"
mstr_url_api  = mstr_base_url+"/MicroStrategyLibrary/api"
project_name  = project_name="MicroStrategy Tutorial"
PROJECT_ID = "B7CA92F04B9FAE8D941C3E9B7E0CD754"

conn=Connection(mstr_url_api, mstr_username, mstr_password, login_mode=1, project_name=project_name)
#env = Environment(connection=conn)

Password ········


# 1. Document      (dossier, document)

In [3]:
from mstrio.project_objects.dossier import list_dossiers
from mstrio.project_objects.document import list_documents

def get_dossier_definition(connection, dossier_id):
    url_add=f"/api/v2/dossiers/{dossier_id}/definition"
    res = connection.get(url=connection.base_url+url_add)
    return res

dossiers_list = list_dossiers(connection=conn, project_name=project_name)
print(f"Number of dossiers found: {len(dossiers_list)}")

documents_list = list_documents(connection=conn, project_name=project_name)
print(f"Number of documents found: {len(documents_list)}")

Number of dossiers found: 71
Number of documents found: 370


In [4]:
documents_all = [['NA', 'NA', 'NA']] 
# this first element with triple "NA" will be used as a dummy parent document by reports and cubes 
# that do not have parent documents

for i, d in enumerate(dossiers_list[:]):
    print(i," / ",len(dossiers_list)," - ", d.name, d.id, d.type.name)
    documents_all.append(["DOSSIER",d.id,d.name])

print("\n\n\nDocuments")
for i, d in enumerate(documents_list[:]):
    print(i," / ",len(documents_list)," - ", d.name, d.id, d.type.name)
    documents_all.append(["DOCUMENT",d.id,d.name])

print("\n",len(documents_all))

0  /  71  -  MicroStrategy, Ink. 0B38895711E79D7ECFF20080AFB1518B DOCUMENT_DEFINITION
1  /  71  -  Campaign Finance 0D0CDC8F11E97E8F00000080AFB3A407 DOCUMENT_DEFINITION
2  /  71  -  Rustic 11EA496611E751FB1E9A0080EF25C29E DOCUMENT_DEFINITION
3  /  71  -  Waterfall 1532D3684B9D88716CFFAA921CA17458 DOCUMENT_DEFINITION
4  /  71  -  Hide Overlapping Markers 1687A36449FBD4F6CD55A987AD3B98F2 DOCUMENT_DEFINITION
5  /  71  -  US Economy Analysis 19A95FA711EC45A93E0B0080AFAB8FDF DOCUMENT_DEFINITION
6  /  71  -  Graphs and Visualizations 27BB740E11E7A7B40A650080EF856B88 DOCUMENT_DEFINITION
7  /  71  -  Investment Firm Dossier 2C68268711E7933800000080AFB1D2B9 DOCUMENT_DEFINITION
8  /  71  -  Maps with Pie and Clusters 316401E111E6BD5BAD440080EFE1DC37 DOCUMENT_DEFINITION
9  /  71  -  Multinational Bank Dossier 32765C5C11E79D5927C10080EF753106 DOCUMENT_DEFINITION
10  /  71  -  Area Chart 3511C9E011E5983A85B30080EFE560D9 DOCUMENT_DEFINITION
11  /  71  -  The 2014 BNY Mellon Boat Race 371E250C11E5BEF

In [5]:
print(len(documents_all))

headers_datasets=["doc_type", "doc_id" , "doc_name"]
df = pd.DataFrame(documents_all, columns = headers_datasets)
df.to_csv(f"L1-documents.csv", index=False, encoding='utf-8', sep=",", 
          quoting=csv.QUOTE_ALL, escapechar="\\", header=True)

442


# 2. Dataset (report, OLAP cube, data import, internal dossier dataset)

There will be 4 types of datasets:
- regular reports
- OLAP cubes
- dossier datasets: data import dataset
- dossier datasets: datasets built from objects within dossiers

There will be two tables:
- one that maps documents with datasets: list_map_doc_dataset
- one that contains details about datasets: list_dataset_details

--TODO
Make sure "report as a filter", "custom subtotals" or any other hidden relations are catched as well

### Reports

In [6]:
from mstrio.project_objects.report import list_reports

reports_list = list_reports(connection=conn, project_name=project_name)
print(len(reports_list))

This (module mstrio.api.metrics) functionality is a work-in-progress. Use it only if you understand the underlying code. It may change in future updates.
773


In [7]:
reports_details=[]
map_reports_NA =[]
errors=[] # some cubes and reports with consolidations will cause errors here

for i,r in enumerate(reports_list[:]):
    print(i,"/",len(reports_list)," - ", r.type.name.upper(), r.id, r.name, r._subtype)
    map_reports_NA.append(["NA", r.id])
    try:
        if r._subtype not in [776,779]: # 779 is data import, 776 is a cube (mstrio fails here for cubes)
            if r.attributes:
                for a in r.attributes:
                    new_attr=[r.type.name.upper(), r.subtype, r.id, r.name, "ATTRIBUTE", a['id'], a['name']]
                    reports_details.append(new_attr)
            if r.metrics:
                for m in r.metrics:
                    new_metric=[r.type.name.upper(), r.subtype, r.id, r.name, "METRIC", m['id'], m['name']]
                    reports_details.append(new_metric)
    except Exception as e:
        errors.append([r.type.name, r._subtype, r.id, r.name])

0  -  REPORT_DEFINITION 13CFD83A458A68655A13CBA8D7C62CD5 01 Basic Report 768
1  -  REPORT_DEFINITION 2840589948B7F9B817CBB18BC32F91E2 02 Filter - Month Report Filter 768
2  -  REPORT_DEFINITION 7A2720FE4B733F5DECBEC184005C1365 03 Limit - Revenue > 320K 768
3  -  REPORT_DEFINITION 2EA7BDCC4217A2552FEB44A4963BB89B 04 Sales Rank 768
4  -  REPORT_DEFINITION 921518204F65BA10C7DB6385201B9359 05 Sort by Revenue Rank 768
5  -  REPORT_DEFINITION 5881BE8D41D8E8BED284FBB403DD51BD 06 Sort by Revenue Rank - Month Report Filter 768
6  -  REPORT_DEFINITION 0335BDE94B5D6FB9BCA94C88950DE7B3 07 Sort by Revenue Rank - Report Limit - Revenue > 320K 768
7  -  REPORT_DEFINITION 8613806F420B8C372EC061BE6BA32A57 08 Sort by Revenue Rank - Report Filter & Report Limit 768
8  -  REPORT_DEFINITION 25747D6A48AEEA264084D1B3076F014F 09a Revenue by Brand 768
9  -  REPORT_DEFINITION 364039B84F4FD8A0A230A29E31AA7F8A 09b Sort by Revenue Rank - Report  Filter - Report as Filter at the Brand Level 768
10  -  REPORT_DEFINI

82  -  REPORT_DEFINITION 85B5D1DD49B0BF0279B6529747E639FB 2 -- Telco Churn Predictor Validation (Ruleset) 768
83  -  REPORT_DEFINITION BDA6FEEC4DC261E9448E0BB14FE43C93 2 -- Telco Churn Predictor Validation (Tree) 768
84  -  REPORT_DEFINITION E881168646E36754DC856298F1460E40 2 -- Telco Churn Predictor Validation (Tree) 768
85  -  REPORT_DEFINITION F2BE6B7C4FD808116A72FA810821D70C 2 -- Web Order Predictor Training Report 769
86  -  REPORT_DEFINITION 68C9C9764B2EFC3E753EABAE4AC9EF12 3.1 Redundant SQL Passes (SQL Global Optimization - Level 0) 770
87  -  REPORT_DEFINITION 0CAAE56C43AE3E7E8D3B8D9875533DA6 3.2 Remove redundant SQL Passes (SQL Global Optimization - Level 1) 770
88  -  REPORT_DEFINITION A971B7AE11D51ED410005B89EBC3E6A4 3a Significant Buyer Retention 768
89  -  REPORT_DEFINITION F0A9CA0211D61FF810007389EBC3E6A4 3b Repeat Buyer Incentive (Automatic Segmentation) 768
90  -  REPORT_DEFINITION 8E3C388D48D65BB75E3AE0A9E0874E9E 3b Repeat Buyer Incentive (Segmentation Prompts) 768
91 

164  -  REPORT_DEFINITION 7516D9F344E3EE2C808FBC8A4B120919 Bar Chart - Monthly revenue by Country 769
165  -  REPORT_DEFINITION 331E36F345FC38E3A32E8189B84D4F3B Bar Chart - Quarterly revenue by Category 769
166  -  REPORT_DEFINITION C8016A53451B562E83337D9BEB392623 Bar Chart - Revenue, Cost and Profit by Region and Quarter 769
167  -  REPORT_DEFINITION ABDDCB1F420024A2168203918FB84136 Bar Chart - Units Sold by Call Center 769
168  -  REPORT_DEFINITION 2744FE904451EE1992D255BCE17359A9 Bar Stacked Chart - Analysis of Shipping Times 769
169  -  REPORT_DEFINITION 53CBD3014CDD1CAA61728C8A61F32F0E Bestseller Item KPI dataset 768
170  -  REPORT_DEFINITION 1022180441E9D1AE0F825B8010502881 Best Selling Items By Category 768
171  -  REPORT_DEFINITION 416F5F234E660DF6E0C6F5A5CC7A1E24 Best Selling Items Details Dataset 768
172  -  REPORT_DEFINITION F7DD6C964F866F9A566ADC8F0E77926B Big Decimal (Discontinued Code - 20 Digits) 768
173  -  REPORT_DEFINITION 689EFFA04A50FF66D47784A0181043CE Blank BigQu

257  -  REPORT_DEFINITION A6FDCFEE43E2F203611BD1B1C83835C8 Customer Income - Dataset 768
258  -  REPORT_DEFINITION 4509EBE24AA4B6EB28AD5DACDC9E8872 Customer Information 768
259  -  REPORT_DEFINITION 93E4EFE94099646B9A740F96637ED578 Customer KPI Report with This Day Filter 768
260  -  REPORT_DEFINITION 671B15174700EC469104ACB1C7F196AE Customer KPI Report with Time-Prompt 768
261  -  REPORT_DEFINITION 028F2A1446B9ACA28C7ED79D75232B21 Customer List 768
262  -  REPORT_DEFINITION D332260E4D0C69A83D6D15BD583812DD Customer Revenue Information 768
263  -  REPORT_DEFINITION C5AFB59445CBEAE4B9DBBC903E5B5999 Customer Sales 768
264  -  REPORT_DEFINITION 38A45012471D28600247089B886A7E73 Customers Analysis 768
265  -  REPORT_DEFINITION 882790F211D3EB22C000B4B2D86C964F Customer Segmentation by Profile 768
266  -  REPORT_DEFINITION EE7C269211D40A3DC000C8906B98494F Customer Segmentation by Value 768
267  -  REPORT_DEFINITION D6811BC345410EAC62BD09BD87472DDD Customers Segmentation by Profile - Set of cu

345  -  REPORT_DEFINITION 3ADA9032480D18E493B88E84CD937B32 Insert Item Order 778
346  -  REPORT_DEFINITION A725CDF04D6CFE04174FBE99584A8DB9 Insert Item Order New 778
347  -  REPORT_DEFINITION 2423EDE5444DC9F20B1E6D94070FDC05 Insert Report 768
348  -  REPORT_DEFINITION 48F05D304066D4D0F6353186AD040CA1 Insert Sales Opportunity 778
349  -  REPORT_DEFINITION 362776FC4FAC612739C2E9ABC6051145 Insert Sales Opportunity 1st Action 778
350  -  REPORT_DEFINITION 34D6AC79420AF8365B176480D411C3A9 Insert Sales Opportunity Action 778
351  -  REPORT_DEFINITION 1501C7F44AEFE6B66FE1BDB1954EF12D Intelligent Cube - All project languages 776
352  -  REPORT_DEFINITION 8CCD8D9D4051A4C533C719A6590DEED8 Intelligent Cube - Drilling outside the cube is disabled 776
353  -  REPORT_DEFINITION 5731DB3F4496967971E88AB7602D63A6 Intelligent Cube - Dynamic Sourcing 776
354  -  REPORT_DEFINITION 95C113424F0DBA3971CE15AE256BFC03 Intelligent Cube - Geography 776
355  -  REPORT_DEFINITION BD91AF40492D2C188240DEAF7D9D1510 I

434  -  REPORT_DEFINITION 1668602D11D5F0A9100081B3A5E8F8A4 Monthly Sales and Margin - Custom Categories 768
435  -  REPORT_DEFINITION 56D3116148577DC68FFC33B4F84B921E Monthly Sales vs LM & LQ (derived metrics) 768
436  -  REPORT_DEFINITION 020E95484AE99C908F11A9ADCA0AFFB0 Monthly Sales vs LM & LY (1:1 transformations) 768
437  -  REPORT_DEFINITION BDE7833B49C24D1370E70D88915C3EF9 Music Sales by Genre and Label 768
438  -  REPORT_DEFINITION 6955DE0E4A19FEE326864D9AF6335BD9 Music Sales - Regional Share by Month 768
439  -  REPORT_DEFINITION EC01E78D48D3A7BA932CC5BDC7939115 New and Lost Customer Counts 768
440  -  REPORT_DEFINITION 88B9A814442DD773E45E11A5773E9F0F Northeast Unit Analysis 768
441  -  REPORT_DEFINITION DE61A9C711D5C6F6C0000F881FDA1A4F Number of Customers by Region 769
442  -  REPORT_DEFINITION 3896EEC040D7DCEF1ED0F29473FEE6B1 Number of Electronics Items per Brand 769
443  -  REPORT_DEFINITION A2B49A3A6E46B699379F309C146BCFC1 Office Royale Customers 781
444  -  REPORT_DEFINI

523  -  REPORT_DEFINITION 429DFDDDA7BC405E918695F9B9634F75 Report Builder 768
524  -  REPORT_DEFINITION 3B75A0454B2D5F9D5A76778723777502 Report Display Options - Alias and Null Values 768
525  -  REPORT_DEFINITION EC8A740D4100ACC5032B29B4338A16D8 Report Filter - Attribute qualification (elements, IDs) and Metric Qualification 768
526  -  REPORT_DEFINITION 9A8B82F843BB8DC82819D1B11A871A9A Report for Dashboard template 768
527  -  REPORT_DEFINITION EC22ED0B4466215298887C91488DC0FF Report from Import Data - CSV File 768
528  -  REPORT_DEFINITION 038F343B4CE68C01103E9B99A952CF9A Report from Import Data - Excel File 768
529  -  REPORT_DEFINITION 4650A5764A18B4EE4BB9F09FFE7D8546 Report from Import Data - SQL Statement 768
530  -  REPORT_DEFINITION 90F0846B4C88836376BF8EBCE82BF604 Report Item Orders 768
531  -  REPORT_DEFINITION BE4036204B90395A2899C98DD7C37202 Report Opportunities 768
532  -  REPORT_DEFINITION BCCB3DBC4FB0CA42622F0A9A783ED826 Report Prompted on Category 768
533  -  REPORT_DE

606  -  REPORT_DEFINITION 039BAEB74B744D32297364863B813F29 Sales by Category, Subcategory, and Item (Aggregate Awareness) 768
607  -  REPORT_DEFINITION 016CB1464A56B21D11AA589964BA98CF Sales by Customer Age 769
608  -  REPORT_DEFINITION AA1441F749668567E6548F8A1F56CE5C Sales by Customer Age 768
609  -  REPORT_DEFINITION EE17720C47B27AECCC6FACB2B649AF3A Sales by Customer Age 768
610  -  REPORT_DEFINITION 16EB1B1147054EDC717581A2F21826AA Sales by Customer Income 768
611  -  REPORT_DEFINITION 7994CF7E46F01DBF1157B0BBC6166A7A Sales by Customer Income 768
612  -  REPORT_DEFINITION 98DE82684414A804E99D69BEDFE03D16 Sales by Customer Income 769
613  -  REPORT_DEFINITION 5EFE920B4BCC7AB26998DDAA6AB8AE43 Sales by Income Bracket (caching disabled) 768
614  -  REPORT_DEFINITION 64819E27493166BAE554E5B571083683 Sales by Income Bracket (caching enabled) 768
615  -  REPORT_DEFINITION 54EAA1C346700FCA27FDF4BB52C07E8D Sales by Payment Type 768
616  -  REPORT_DEFINITION 1A0E36AA42A062734A8CB2BE28EFB2C8 

695  -  REPORT_DEFINITION D174C8D74D8553B62D1989825BF1E74C Support Dashboard - KPIs performance - % Critical Cases - Year 769
696  -  REPORT_DEFINITION 27DCFBCB4EC92A2827D7EF9AD15DB45F Support Dashboard - KPIs performance - % Customer satisfaction - Month 769
697  -  REPORT_DEFINITION B654A41E491CAA797F06F08172D73D43 Support Dashboard - KPIs performance - % Customer satisfaction - Year 769
698  -  REPORT_DEFINITION 42003B534160D74BD70EA49AB99C5301 Support Dashboard - Prod. Adoption Milestones 768
699  -  REPORT_DEFINITION B9DC123C49D2221184570283FB0344C6 Support Dashboard - Product Adoption Cumulated 769
700  -  REPORT_DEFINITION 8DD2BA1E44BE56A26F2C5CA249021C5E Support Dashboard - Speed of adoption by Product version 769
701  -  REPORT_DEFINITION 5A9B897A4737BD227FC9329A516E481E Telco Churn Contract Details Dataset 768
702  -  REPORT_DEFINITION BD94C2BA4DB5563E95AFF5B0F1B7EFF4 Telco Churn Dataset 768
703  -  REPORT_DEFINITION F29417C611D60AD910008BB3D1CEE6A4 Time Analysis 768
704  -  

In [8]:
# Reports with consolidations fail (REST API problem), there might be other reasons of failure
print("\nERRORS:")
for e in errors:
    print(e)


ERRORS:
['REPORT_DEFINITION', 768, 'FA5C5E7345A654AFD9E4FBAE6CA5E802', '2- Category Performance Summary by Region']
['REPORT_DEFINITION', 768, 'EC69D9F74AC0077CDE5A2C91324A616C', '3 -- Customer Segment Analysis']
['REPORT_DEFINITION', 768, '69BDD86844177CF0882E9F8BB6F03220', '4- Category Performance Detail by Region']
['REPORT_DEFINITION', 768, '8AA156484B970B22EB1B0A86717DD3EA', 'Customer Geography']
['REPORT_DEFINITION', 768, 'E472644E4239AD7EFDBD869600C93E4D', 'Customer Geography Ratios']
['REPORT_DEFINITION', 768, 'EE7C269211D40A3DC000C8906B98494F', 'Customer Segmentation by Value']
['REPORT_DEFINITION', 769, 'C77F992D11D60B4BC0008F916B98494F', 'Decile Histogram - Graph']
['REPORT_DEFINITION', 768, '57ACE37343608850FAEC918318843B14', 'Decile Histogram - Grid']
['REPORT_DEFINITION', 768, 'DB8D5B064BBE3C24F541DAA81A507FDC', 'Intelligent Cube Report']
['REPORT_DEFINITION', 768, '1668602D11D5F0A9100081B3A5E8F8A4', 'Monthly Sales and Margin - Custom Categories']
['REPORT_DEFINITION', 7

In [ ]:
# TODO
# use search to find component objects for those reports from error list

In [ ]:
def get_report_components(PROJECT_ID, rep_ID):
    doc_objects=[]
    objects = full_search(
        conn,
        project=PROJECT_ID,
        used_by_object_id =rep_ID,
        used_by_object_type =ObjectTypes.REPORT_DEFINITION,
        #object_types=[ObjectTypes.REPORT_DEFINITION],
        used_by_recursive =False
    )
    for ob in objects:
        if ob['type']==12 or (ob['type'] == 4 and ob['subtype'] == 1024):
            elements=[doc_ID,ob["id"],ob['name'],ob['type'],ob['subtype']]
            print(elements)
            doc_objects.append(elements)
    return doc_objects

In [ ]:
do=get_report_components(PROJECT_ID, "EC69D9F74AC0077CDE5A2C91324A616C")

In [ ]:
# TODO - make sure bulk reports are processed correctly

### OLAP Cubes

In [9]:
from mstrio.project_objects import OlapCube, Report
from mstrio.project_objects.datasets.olap_cube import list_olap_cubes
olap_list = list_olap_cubes(connection=conn, project_name=project_name)
print(len(olap_list))

15


In [10]:
# get components of each OLAP cube
olap_details=[]
map_olap_NA =[]

for olap in olap_list[:]:
    map_olap_NA.append(["NA", olap.id])
    for a in olap.attributes:
        olapc=["OLAPCUBE",olap.subtype,olap.id,olap.name, "ATTRIBUTE", a["id"], a["name"]]
        print(olapc)
        olap_details.append(olapc)
    for m in olap.metrics:
        olapc=["OLAPCUBE",olap.subtype,olap.id,olap.name, "METRIC", m["id"], m["name"]]
        print(olapc)
        olap_details.append(olapc)
        
print("\n\n",len(olap_details))

['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Performance Cube', 'ATTRIBUTE', '8D679D4411D3E4981000E787EC6DE8A4', 'Month']
['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Performance Cube', 'ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year']
['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Performance Cube', 'ATTRIBUTE', '8D679D3511D3E4981000E787EC6DE8A4', 'Call Center']
['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Performance Cube', 'ATTRIBUTE', '8D679D4B11D3E4981000E787EC6DE8A4', 'Region']
['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Performance Cube', 'ATTRIBUTE', '8D679D3711D3E4981000E787EC6DE8A4', 'Category']
['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Performance Cube', 'ATTRIBUTE', '8D679D4F11D3E4981000E787EC6DE8A4', 'Subcategory']
['OLAPCUBE', 776, 'CC02C5C24AE2803ABF14EDA5038159D4', 'Actual vs Forecast Perfo

['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic Sourcing', 'ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year']
['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic Sourcing', 'ATTRIBUTE', '8D679D4A11D3E4981000E787EC6DE8A4', 'Quarter']
['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic Sourcing', 'ATTRIBUTE', '8D679D4411D3E4981000E787EC6DE8A4', 'Month']
['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic Sourcing', 'ATTRIBUTE', '8D679D3711D3E4981000E787EC6DE8A4', 'Category']
['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic Sourcing', 'ATTRIBUTE', '8D679D4F11D3E4981000E787EC6DE8A4', 'Subcategory']
['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic Sourcing', 'ATTRIBUTE', '8D679D5011D3E4981000E787EC6DE8A4', 'Supplier']
['OLAPCUBE', 776, '5731DB3F4496967971E88AB7602D63A6', 'Intelligent Cube - Dynamic

['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D3711D3E4981000E787EC6DE8A4', 'Category']
['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D4F11D3E4981000E787EC6DE8A4', 'Subcategory']
['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D5011D3E4981000E787EC6DE8A4', 'Supplier']
['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year']
['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D4A11D3E4981000E787EC6DE8A4', 'Quarter']
['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D4411D3E4981000E787EC6DE8A4', 'Month']
['OLAPCUBE', 776, '082E2B4A4D65C28B998A5785431B8729', 'Intelligent Cube - Inventory', 'ATTRIBUTE', '8D679D4B11D3E4981000E78

['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D4111D3E4981000E787EC6DE8A4', 'Income Bracket']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '6E069E4C11D3E4E41000E887EC6DE8A4', 'Customer Age']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D3C11D3E4981000E787EC6DE8A4', 'Customer']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D4A11D3E4981000E787EC6DE8A4', 'Quarter']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D4411D3E4981000E787EC6DE8A4', 'Month']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D3511D3E4981000E787EC6DE8A4', 'Call Center']
['OLAPCUBE', 776, 'DF05CEA04CCA432397E466A55F1A9EC7', 'Order Cube', 'ATTRIBUTE', '8D679D4B11

### Initial list datasets_NA

In [11]:
# we treat them (reports + olap cubes) as NA - not assigned at this point
# at later stage we will remove from this list datasets that are used by dossiers/documents
# only truly unasigned datasets will remained marked as NA
# each row contains information about dataset ID and single component: either metric or attribute

# lets merge two sources: reports and olap cubes
map_doc_dataset_NA=map_reports_NA + map_olap_NA
dataset_details=reports_details + olap_details

print("Number of collected NA datasets:",len(map_doc_dataset_NA))
print("Number of rows per dataset (attributes and metrics components):", len(dataset_details))

Number of collected NA datasets: 788
Number of rows per dataset (attributes and metrics components): 4037


## Dossiers and Documnets

### Dossiers

In [12]:
map_dossier = [] # this will be mapping dossier/document with their datasets
dossier_details = [] # this will collect details of each dataset

In [13]:
for i, d in enumerate(dossiers_list[:]):
    print(i,"/", len(dossiers_list), " - ", d.name, d.id, d.type.name)
    r=get_dossier_definition(conn, d.id).json()
    for dset in r['datasets']:
        map_dossier.append([d.id, dset['id']])
        for ao in dset['availableObjects']:
            dossier_details.append(["DATASET", "DATASET", dset['id'], dset['name'], ao['type'].upper(), ao['id'], ao['name']])

0 / 71  -  MicroStrategy, Ink. 0B38895711E79D7ECFF20080AFB1518B DOCUMENT_DEFINITION
1 / 71  -  Campaign Finance 0D0CDC8F11E97E8F00000080AFB3A407 DOCUMENT_DEFINITION
2 / 71  -  Rustic 11EA496611E751FB1E9A0080EF25C29E DOCUMENT_DEFINITION
3 / 71  -  Waterfall 1532D3684B9D88716CFFAA921CA17458 DOCUMENT_DEFINITION
4 / 71  -  Hide Overlapping Markers 1687A36449FBD4F6CD55A987AD3B98F2 DOCUMENT_DEFINITION
5 / 71  -  US Economy Analysis 19A95FA711EC45A93E0B0080AFAB8FDF DOCUMENT_DEFINITION
6 / 71  -  Graphs and Visualizations 27BB740E11E7A7B40A650080EF856B88 DOCUMENT_DEFINITION
7 / 71  -  Investment Firm Dossier 2C68268711E7933800000080AFB1D2B9 DOCUMENT_DEFINITION
8 / 71  -  Maps with Pie and Clusters 316401E111E6BD5BAD440080EFE1DC37 DOCUMENT_DEFINITION
9 / 71  -  Multinational Bank Dossier 32765C5C11E79D5927C10080EF753106 DOCUMENT_DEFINITION
10 / 71  -  Area Chart 3511C9E011E5983A85B30080EFE560D9 DOCUMENT_DEFINITION
11 / 71  -  The 2014 BNY Mellon Boat Race 371E250C11E5BEFB00000080EF85899C DOCUME

In [14]:
# remove duplicates
import itertools
map_dossier.sort()
map_dossier=list(map_dossier for map_dossier,_ in itertools.groupby(map_dossier))

dossier_details.sort()
dossier_details=list(dossier_details for dossier_details,_ in itertools.groupby(dossier_details))

In [15]:
print("Number of collected dossier datasets:",len(map_dossier))
print("Number of rows per dossier dataset (attributes and metrics components):", len(dossier_details))

Number of collected dossier datasets: 165
Number of rows perdossier dataset (attributes and metrics components): 1902


#### Documents

In [16]:
map_document = [] # this will be mapping dossier/document with their datasets
document_details = [] # this will collect details of each dataset

In [17]:
def get_document_datasets(PROJECT_ID, doc_ID):
    map_datasets, datasets_details=[],[]
    datasets = full_search(
        conn,
        project=PROJECT_ID,
        used_by_object_id =doc_ID,
        used_by_object_type =ObjectTypes.DOCUMENT_DEFINITION,
        object_types=[ObjectTypes.REPORT_DEFINITION],
        used_by_recursive =False
    )
    for dset in datasets:
        map_datasets.append([doc_ID,dset["id"]])
        if dset['type']==3:            
            elements=["DATASET_NA", "DATASET_NA", dset["id"],dset['name'],"NA","NA","NA"] # ob['type'],ob['subtype'],
            print(elements)
            datasets_details.append(elements)
    return map_datasets, datasets_details

In [21]:
%%time
for i,doc in enumerate(documents_list[:]):
    print(i, "/", len(documents_list), " - ", doc.id, doc.name)
    # collecting dataset_details for documents is probablu unnecessery
    # documents can only contain regular reports or cubes (olap or imported)
    # details of those were collected in previous steps
    map_datasets, datasets_details = get_document_datasets(PROJECT_ID,doc.id)
    document_details.extend(datasets_details)
    map_document.extend(map_datasets)

0 / 370  -  009DDDBA43B89B78FC2837810239D316 iPad Retina Landscape
1 / 370  -  010594C111E58D3500000080EF45B567 Category Management Dashboard
['DATASET_NA', 'DATASET_NA', '1F8112B441C91B565A7A54ACA2440F55', 'Category Performance Dataset', 'NA', 'NA', 'NA']
2 / 370  -  01FEBE9011E6540100000080EF451B8D Import .sas7bdat files
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
3 / 370  -  036C946741BB821F9DE3BE82C19FA45A iPad Landscape
4 / 370  -  04B7923744F70400A0FAE4BE3C1809B3 8. Nearby Stores
['DATASET_NA', 'DATASET_NA', 'E47A4E374E45FC60BEEEA1B85560EE48', 'Stores within x miles', 'NA', 'NA', 'NA']
5 / 370  -  05155F1A471F49B5DA33F7AB87188923 Geography Hierarchy
6 / 370  -  06E1AE0311E608AE27580080EFE532B4 Hide Overlapping Data Labels
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
7 / 370  -  078317FA437448F32CCFC5B7056C7B84 Operational Performan

37 / 370  -  196956D04B7F6930DD39079513187BB4 Custom Formatting
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
38 / 370  -  1AC0D99E44EC77B246D216A2341F4254 Data Wrangling
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
39 / 370  -  1AC2014C4A575085AD7809A3F8D075F9 Operational Performance Scorecard
['DATASET_NA', 'DATASET_NA', '13CDFAD64BA03B5ECE7B1185C32F55AC', 'Avg Revenue per Customer Analysis', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '4008F79B41B208BEF567E0AB99C13B06', 'Avg Revenue per Order Analysis', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '38A45012471D28600247089B886A7E73', 'Customers Analysis', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '2988553D43B5757134D3A1814A8F5A34', 'Margin Analysis', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '5E4835EA468983EFD4855FB15CEE727C', 'Month', 'NA', 'NA', 'NA']
['DATASET_NA', 'DAT

78 / 370  -  3E13CE2748127AC48E230AB03C1F3CBB Import Data From JSON
['DATASET_NA', 'DATASET_NA', 'C509EF2042FA23C06332E481319E22E7', 'Consumer Complaints (JSON)', 'NA', 'NA', 'NA']
79 / 370  -  3EDDBBBB4A242F68CF4E649F7A55F618 Reports
['DATASET_NA', 'DATASET_NA', 'E659E86811E58C918D6F0080EF453539', 'Simple Report', 'NA', 'NA', 'NA']
80 / 370  -  3EE7568111E65331101B0080EFD53B8C Improved Axis Labels
['DATASET_NA', 'DATASET_NA', 'E659E86811E58C918D6F0080EF453539', 'Simple Report', 'NA', 'NA', 'NA']
81 / 370  -  407236B74140555B8D37D7BDDBBAD83C Store Details
['DATASET_NA', 'DATASET_NA', 'EA32F0514BEEADF987C9C59C3DE5733B', 'Linked Store Details', 'NA', 'NA', 'NA']
82 / 370  -  4186A0C74AC59879271BBEA8978406F2 Monochrome
83 / 370  -  41B1166B4C84A759CC2C19BB07C2EC00 Inventory by Subcategory
['DATASET_NA', 'DATASET_NA', 'E98CCCFD4827A646B162208AB3763D18', 'Inventory by Item', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', 'FC1394F4422E1EDCC11D0F98F5D33A43', 'Inventory Summary by Subcategory'

['DATASET_NA', 'DATASET_NA', '064C5F9E4292E03172A2B390F3C9201A', 'MAP MULTI Cost Across States', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '19C83B3A441EE255D2A6AD94443B12C4', 'MAP MULTI Data Per Supplier', 'NA', 'NA', 'NA']
113 / 370  -  5A9C8A0B411B76F5FFAD8EB4DC8BE7AD Multi-Layer Maps for iPad
['DATASET_NA', 'DATASET_NA', '064C5F9E4292E03172A2B390F3C9201A', 'MAP MULTI Cost Across States', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '19C83B3A441EE255D2A6AD94443B12C4', 'MAP MULTI Data Per Supplier', 'NA', 'NA', 'NA']
114 / 370  -  5B8142D611E5FB13FE000080EFE5AFA8 Data Assets
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
115 / 370  -  5C88F96F45F011C53088A488FC86A08E Import Reports and Dashboards (with formatting)
116 / 370  -  5CB7868D4A2BFDC4A1B85CB3A209DCB2 Import Reports and Dashboards (with formatting)
117 / 370  -  5CF40238425FA1A251ACF9925F846B62 04 Title and Two Contents
118 / 370  -  5EB336634F3C9

['DATASET_NA', 'DATASET_NA', 'A20C8AC711D60AE310008BB3D1CEE6A4', 'Regional Profit and Margins', 'NA', 'NA', 'NA']
166 / 370  -  78DB7F8411E6540200000080EF15BB8C Import .sas7bdat files
['DATASET_NA', 'DATASET_NA', 'E659E86811E58C918D6F0080EF453539', 'Simple Report', 'NA', 'NA', 'NA']
167 / 370  -  79B8D67B4315A33F7E209BB216DE3C21 iPhone Landscape Micro-Application
168 / 370  -  7A58E70D11E5D0D60C470080EFE5C0B6 What's New in MicroStrategy 10
169 / 370  -  7AD4832811E6533800000080EF451C8F Improved Axis Labels
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
170 / 370  -  7B19124E4C3ED474F6C340A37C6365C1 Enhanced ESRI Maps for Android
['DATASET_NA', 'DATASET_NA', '4BA8E05D4FEE31067E66C1A0C6FB5088', 'State Store Performance', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '11DE1AF64D2AB24D15481593136F8A15', 'Top 100 US Stores', 'NA', 'NA', 'NA']
171 / 370  -  7B2607A246FEFD21BACED58306DB6162 9 Financial Analysis
172 / 37

['DATASET_NA', 'DATASET_NA', '03981D054654EB41C053889A967429CE', 'Store Gauges (Prompted)', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', 'B532EDDD41A9942934AF46994C1B0160', 'Store Level Dataset (Prompted)', 'NA', 'NA', 'NA']
206 / 370  -  932B2F634C846A9FFAEC77A2D986CCFA Visualizations
['DATASET_NA', 'DATASET_NA', 'E659E86811E58C918D6F0080EF453539', 'Simple Report', 'NA', 'NA', 'NA']
207 / 370  -  933BC42947828BD2BC1629B914B9C9F5 Blue and Grey
208 / 370  -  93A590F149167B1133C9E882472103E8 Advanced Analytics and Data Mining
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
209 / 370  -  93D8F52A414E757B9A6EA1BEF580FC35 3 Statement Credit Card
210 / 370  -  9445BA9E4276289D7676418A7BB58B3A 14 Standard BI Template
211 / 370  -  965C9529499647BE0C2AF2846917E6EB Category Performance Dashboard
['DATASET_NA', 'DATASET_NA', 'E109F3C8439CE1F6A53DDDA7E8AD9EBE', 'Category Measures', 'NA', 'NA', 'NA']
212 / 370  -  97FC5CCB4C

['DATASET_NA', 'DATASET_NA', '448C55AF4461BC616DFED5B49C0593AE', 'Call Center Manager List', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', 'E05691E74D69F36181DEFD9F084D632C', 'Churner Demographics', 'NA', 'NA', 'NA']
251 / 370  -  B23339FA450D6AA3091B3B9FCEAA53FD Custom Formatting
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
252 / 370  -  B2901DB511E568EB20E80080EFB55291 Operational Performance Dashboard
['DATASET_NA', 'DATASET_NA', '97928D744AEFCB7EF3087CA52E62944D', 'KPI List', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '6A0449B940FC78174864E5B4C28BA721', 'KPI List - Region and Category', 'NA', 'NA', 'NA']
253 / 370  -  B29FE68846415377EE6E8FAA5D90BDAC Direct Data Access
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
254 / 370  -  B370AC57461D352C1D77CFBE2C150C7E Sales by Customer State
['DATASET_NA', 'DATASET_NA', 'F47A8DE94FB0472EE

['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
294 / 370  -  C7BFF50F46307810DC92C3BBB5064833 Promotions Hierarchy
295 / 370  -  C7F3D46E11E5C60827AF0080EFB56EC4 Company Sales
['DATASET_NA', 'DATASET_NA', '77AEF97745A220B21A1F3EB6B00364E0', 'Customer Details', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '39B64D784FD0ABD300A444B088F74986', 'Heat Map Widget', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '3C46FF784B658D4DD346A89B3304B3F3', 'KPIs ThisMonth', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '8174596048F5AD7EED41E896D07E29D9', 'KPIs ThisYear', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '8BC8D9A4419B6747D74432B487C8A305', 'KPIs Today', 'NA', 'NA', 'NA']
296 / 370  -  C87490B1453C9225362ED4895FBB9FF1 Embedded Images
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
297 / 370  -  C9086B8649B8BDD9036B65A1160DE12E Visualizations
['D

336 / 370  -  E22ECEF34E4314F9F5744786E9453B89 Preferred Data Import Connections
337 / 370  -  E25785B74325E3EE3BADED9FE1E39C6D Red and Black
338 / 370  -  E44B880748568EA00393E1BB3ED262D0 05 Two Titles and Two Contents
339 / 370  -  E594227A40203E702B87EFA04BF45A47 Attribute Object Selector
['DATASET_NA', 'DATASET_NA', 'D825F0E211E5D0350A580080EFD5D8BC', 'Attribute Object Selector Dataset', 'NA', 'NA', 'NA']
340 / 370  -  E5F3F5914BC1E973AE03119035D14FA1 iPhone Portrait Micro-Application
341 / 370  -  E6C01C0011E5725B1C520080EF45718E Tutorial Home
342 / 370  -  E6E6D483411DB3CF5BCFE894D134EE9D Category Sales & Profitability Dashboard
['DATASET_NA', 'DATASET_NA', 'B7E14BD6439AC1F680AD298860D1F16D', 'Executive Business User Data', 'NA', 'NA', 'NA']
['DATASET_NA', 'DATASET_NA', '5B65EEBD4B725806350A88B89DE99088', 'Subcategory by Region Revenue Analysis', 'NA', 'NA', 'NA']
343 / 370  -  E7E92BAC11E5D1020AA50080EF45F98C Hide Overlapping Data Labels
344 / 370  -  E9E9295311E5CDE200000080EFA

In [22]:
# remove duplicates
import itertools
map_document.sort()
map_document=list(map_document for map_document,_ in itertools.groupby(map_document))

document_details.sort()
document_details=list(document_details for document_details,_ in itertools.groupby(document_details))

In [23]:
print("Number of collected dossier datasets:",len(map_document))
print("Number of rows per dossier dataset (attributes and metrics components):", len(document_details))

Number of collected dossier datasets: 430
Number of rows per dossier dataset (attributes and metrics components): 200


### Merge multiple lists

#### mapping

In [24]:
# dossier and document datasets have priority
mapped_datasets_main=map_dossier+map_document
mapped_datasets_main_ids=[m[1] for m in mapped_datasets_main]

print("Before:",len(mapped_datasets_main))

# NA datasets will be added to the main list only if they are not already there
for md in map_doc_dataset_NA:
    if md[1] not in mapped_datasets_main_ids:
        mapped_datasets_main.append(md) # this is a report or olap cube that is not used by dossier or document

print("After:",len(mapped_datasets_main))

Before: 595
After: 1172


#### details

In [25]:
# we have an opposite situation here. rows from NA datasets have priority
# they contain info about dataset subtype
doc_dataset_details=document_details+dossier_details
dataset_details_ids=[dset[2] for dset in dataset_details]
dataset_details_ids=list(set(dataset_details_ids)) #remove duplicates

print("Before:",len(dataset_details))

# NA datasets will be added to the main list only if they are not already there
for dset in doc_dataset_details:
    if dset[2] not in dataset_details_ids:
        dataset_details.append(dset) # this is a report or olap cube that is not used by dossier or document

print("After:",len(dataset_details))

Before: 4037
After: 5885


### Export the final lists

In [26]:
headers_datasets=["doc_id","dataset_id"]
df = pd.DataFrame(mapped_datasets_main, columns = headers_datasets)
df.to_csv(f"L1a-mapping_document_dataset.csv", index=False, encoding='utf-8', sep=",", 
          quoting=csv.QUOTE_ALL, escapechar="\\", header=True)

In [27]:
headers_datasets=["dataset_type","dataset_subtype","dataset_id","dataset_name",
                  "repobj_type","repobj_id","repobj_name"]
df = pd.DataFrame(dataset_details, columns = headers_datasets)
df.to_csv(f"L2-datasets.csv", index=False, encoding='utf-8', sep=",", 
          quoting=csv.QUOTE_ALL, escapechar="\\", header=True)

# 4-6. attributes, facts, tables, data sources

#### Iterate through tables. For each table search for attributes and facts

In [28]:
%%time
from mstrio.modeling.schema.table import list_logical_tables

tables_list = list_logical_tables(connection=conn)
data_list=[]

for i, table in enumerate(tables_list[:]):
    print("\n", i," / ",len(tables_list)," >>> ===========")
    print(table.name, table.id, table.subtype, table.primary_data_source.name)
    print("===========")
    if table.attributes and table.subtype==3840:
        for a in table.attributes:
            if a.id  and a.sub_type=="attribute": # and a.subtype==3072
                aa=a.list_properties() ## Important! it downloads all the details
                print("\n-- ", a.name, a.id, a.subtype, a.sub_type) 
                for form in a.forms:
                    if not form.is_form_group: # eliminate grouped forms
                        form_name=form.name
                        print("### ", form_name, form.data_type)
                        if form.data_type:
                            form_datatype,form_precis=form.data_type.type, form.data_type.precision
                            
                            for expr in form.expressions:
                                expr_text = expr.expression.text
                                data_list.append([table.name, table.id,table.primary_data_source.name,
                                                  a.type.name, a.id, a.name, 
                                                  a.attribute_lookup_table.name, form_name,form_datatype,
                                                  form_precis, expr.expression.text])                            
                        else:
                            form_datatype,form_precis,expr_text="NA","NA","NA"
                            data_list.append([table.name, table.id,table.primary_data_source.name,
                                              a.type.name, a.id, a.name, 
                                              a.attribute_lookup_table.name, form_name,form_datatype,
                                              form_precis, expr.expression.text])
    if table.facts:
        for f in table.facts:            
            if f.id:
                print("\nFACTS")
                for expr in f.expressions:
                    # facts do not have forms - add NA instead
                    data_list.append([table.name, table.id,table.primary_data_source.name,
                                      f.type.name, f.id, f.name,
                                      "NA", "NA",f.data_type.type,
                                      f.data_type.precision, expr.expression.text])                    



 0  /  73  >>> ===========
PMT_INVENTORY 8D67932011D3E4981000E787EC6DE8A4 3842 Tutorial Postgres

 1  /  73  >>> ===========
INVENTORY_ORDERS 8D67932C11D3E4981000E787EC6DE8A4 3840 Tutorial Postgres

--  Item 8D679D4211D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Long Desc DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Foreign Name DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Price DataType(type='decimal', precision=0, scale=-2147483648)
###  Image DataType(type='n_var_char', precision=250, scale=0)
###  Barcode DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Barcode Image DataType(type='integer', precision=2, scale=0)

--  Month 8D679D4411D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(


--  Phone Usage 95FA2DC34EC9063FD6C400B4FA8570E0 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  Active Months DataType(type='decimal', precision=0, scale=-2147483648)
###  Average Minutes Off-peak DataType(type='decimal', precision=0, scale=-2147483648)
###  Average Minutes Peak DataType(type='decimal', precision=0, scale=-2147483648)
###  Cancel Date DataType(type='time_stamp', precision=8, scale=-2147483648)
###  Churn DataType(type='integer', precision=2, scale=-2147483648)
###  Current Months DataType(type='decimal', precision=0, scale=-2147483648)
###  Dropped Calls DataType(type='integer', precision=2, scale=-2147483648)
###  Expire Date DataType(type='time_stamp', precision=8, scale=-2147483648)
###  Helpdesk Calls DataType(type='integer', precision=2, scale=-2147483648)
###  Lifetime Value DataType(type='decimal', precision=0, scale=-2147483648)
###  Monthly Profit DataType(type='decimal', precision=0, scale=-2147483648)
###  Net Present V


--  MicroStrategy User 2501818A4033B34A5FCD3FBCC8F7F30F 3072 attribute
###  ID DataType(type='fixed_length_string', precision=50, scale=-2147483648)

 14  /  73  >>> ===========
LU_MONTH 8D67938011D3E4981000E787EC6DE8A4 3840 Tutorial Postgres

--  Month 8D679D4411D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  DATE DataType(type='time_stamp', precision=8, scale=-2147483648)

--  Month of Year 8D679D4511D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=50, scale=-2147483648)

--  Quarter 8D679D4A11D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  DATE DataType(type='time_stamp', precision=8, scale=-214748


--  Item 8D679D4211D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Long Desc DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Foreign Name DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Price DataType(type='decimal', precision=0, scale=-2147483648)
###  Image DataType(type='n_var_char', precision=250, scale=0)
###  Barcode DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Barcode Image DataType(type='integer', precision=2, scale=0)

--  Employee 8D679D3F11D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  Last Name DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  First Name DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  SSN DataType(type='fixed_length_string'

RUSH_ORDER 8D6793F211D3E4981000E787EC6DE8A4 3840 Tutorial Postgres

--  Order 8D679D4611D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  Date DataType(type='time_stamp', precision=8, scale=-2147483648)

--  Rush Order A2AFB70811D3E4A81000E787EC6DE8A4 3072 attribute
###  ID DataType(type='fixed_length_string', precision=50, scale=-2147483648)

FACTS

 31  /  73  >>> ===========
YR_CATEGORY_SLS 8D67941611D3E4981000E787EC6DE8A4 3840 Tutorial Postgres

--  Category 8D679D3711D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Image DataType(type='n_var_char', precision=50, scale=0)

--  Year 8D679D5111D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DATE DataType(type='time_stamp', precision=8, scale=-2147483648)

FACTS

FACTS

FACTS

FACTS


FACTS

FACTS

FACTS

 42  /  73  >>> ===========
ITEM_MNTH_SLS 929118964740ECCB9DB121A0A60A78B1 3840 Tutorial Postgres

--  Item 8D679D4211D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Long Desc DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Foreign Name DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Price DataType(type='decimal', precision=0, scale=-2147483648)
###  Image DataType(type='n_var_char', precision=250, scale=0)
###  Barcode DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Barcode Image DataType(type='integer', precision=2, scale=0)

--  Month 8D679D4411D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  DATE DataType

LU_MARITALSTATUS 0139A31144A93D3FD7216CA4ADBD404F 3840 Tutorial Postgres

--  Marital Status 846E77494CAFEAAC43BA51986F6EBCBF 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=20, scale=-2147483648)

 58  /  73  >>> ===========
LU_AGERANGE ED8A34EC483EA8946D2A248FEF9A116B 3840 Tutorial Postgres

--  Age Range 5603951E4FE1BC04A44E44B85BBB8ED2 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=20, scale=-2147483648)

 59  /  73  >>> ===========
LU_GENDER D953F9044B9181FB6E0C0090644F991C 3840 Tutorial Postgres

--  Customer Gender 4151F1304F00A51EB13BCF84283396BE 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=20, scale=-2147483648)

 60  /  73  >>> ===========
LU_EDUCATION 774F53FD4290CBB76FC95EAE70D4A532 3840 Tutorial Postgres

-

ITEM_CCTR_MNTH_SLS 980A427E49EA03F02C85E2A95C66F794 3840 Tutorial Postgres

--  Item 8D679D4211D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=2, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Long Desc DataType(type='fixed_length_string', precision=255, scale=-2147483648)
###  Foreign Name DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Price DataType(type='decimal', precision=0, scale=-2147483648)
###  Image DataType(type='n_var_char', precision=250, scale=0)
###  Barcode DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  Barcode Image DataType(type='integer', precision=2, scale=0)

--  Month 8D679D4411D3E4981000E787EC6DE8A4 3072 attribute
###  ID DataType(type='integer', precision=4, scale=-2147483648)
###  DESC DataType(type='fixed_length_string', precision=50, scale=-2147483648)
###  DATE DataType(type='time_stamp', precision=8, scale=-21474

In [29]:
# Export to csv
headers_datasets=["tbl_name", "tbl_id" , "tbl_data_source", "schemaobj_type", "schemaobj_id", "schemaobj_name", 
                  "attr_lu_table", "form_name", "form_datatype", "form_precision", "expression"]
df = pd.DataFrame(data_list, columns = headers_datasets)
df.to_csv("L456-Objects.csv", index=False, encoding='utf-8', sep=",", 
          quoting=csv.QUOTE_ALL, escapechar="\\", header=True)

### generate list of attributes to use as report objects

In [30]:
# Attribute will have itself as a component object
# Higher level attribute will act as a report object just as metric
# Example: ['ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year', 'ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year']
list_attr_rep_obj=[]
for obj in data_list:
    if obj[3] == "ATTRIBUTE":
        attr=[obj[3],obj[4],obj[5],obj[3],obj[4],obj[5]]
        if attr not in list_attr_rep_obj:
            list_attr_rep_obj.append(attr)
            print(attr)

['ATTRIBUTE', '8D679D4211D3E4981000E787EC6DE8A4', 'Item', 'ATTRIBUTE', '8D679D4211D3E4981000E787EC6DE8A4', 'Item']
['ATTRIBUTE', '8D679D4411D3E4981000E787EC6DE8A4', 'Month', 'ATTRIBUTE', '8D679D4411D3E4981000E787EC6DE8A4', 'Month']
['ATTRIBUTE', '8D679D3511D3E4981000E787EC6DE8A4', 'Call Center', 'ATTRIBUTE', '8D679D3511D3E4981000E787EC6DE8A4', 'Call Center']
['ATTRIBUTE', '8D679D3811D3E4981000E787EC6DE8A4', 'Country', 'ATTRIBUTE', '8D679D3811D3E4981000E787EC6DE8A4', 'Country']
['ATTRIBUTE', '8D679D4311D3E4981000E787EC6DE8A4', 'Manager', 'ATTRIBUTE', '8D679D4311D3E4981000E787EC6DE8A4', 'Manager']
['ATTRIBUTE', '8D679D4B11D3E4981000E787EC6DE8A4', 'Region', 'ATTRIBUTE', '8D679D4B11D3E4981000E787EC6DE8A4', 'Region']
['ATTRIBUTE', '8D679D5211D3E4981000E787EC6DE8A4', 'Distribution Center', 'ATTRIBUTE', '8D679D5211D3E4981000E787EC6DE8A4', 'Distribution Center']
['ATTRIBUTE', '8D679D3611D3E4981000E787EC6DE8A4', 'Catalog', 'ATTRIBUTE', '8D679D3611D3E4981000E787EC6DE8A4', 'Catalog']
['ATTRIBUTE'

# Report objects: metrics (+ attributes)

In [ ]:
# TODO
# make sure relations to attributes and facts via levels, filters, transformations and so on are also captured

In [31]:
from mstrio.modeling import (list_metrics, Metric)
from mstrio.modeling.expression import Expression, Token

# Some functions
# metrics and base formulas must be treated separately

def get_metric_definition(metric_id, metric_name):
    oc=[]
    try:
        metric = Metric(connection=conn, id=metric_id)       
        try:
            for c in metric.expression.tokens:
                if c.type.value in ["object_reference",'object_at_form']:
                    metric_definition=[metric.sub_type.value.upper(), metric_id, metric_name,
                                       c.target.sub_type.name.upper(),c.target.object_id,c.value]
                    if metric_definition not in oc:
                        oc.append(metric_definition)
        except:
            print("### ERROR 2 - no valid Expression ", metric_id, "\n")
    except:
        print("### ERROR 1 - not a Metric ", metric_id, "\n")

    return oc

def get_base_formula_definition(connection, formula_id, formula_name):
    url_add=f"/api/model/formulas/{formula_id}?showExpressionAs=tokens"
    res = connection.get(url=connection.base_url+url_add)
    tokens=res.json()["expression"]["tokens"]
    bf_comps=[]
    for token in tokens:
        if token['type'] in ["object_reference",'object_at_form']:
            bformula=["AGG_METRIC",formula_id, formula_name,
                      token['target']['subType'].upper(),
                      token['target']['objectId'],                      
                      token['target']['name']]
            bf_comps.append(bformula)    
    return bf_comps

def dig_metric(m, oc, metrics_to_resolve, newly_found):
    # For scenario when metric is based on other metrics recursively
    # 
    corr_oc=[] # replacing parent object with the initial metric
    for o in oc:
        print(o)
        corr_obj=[m[0],m[1],m[2],o[3],o[4],o[5]]
        corr_oc.append(corr_obj)
        # find new metrics to be resolved
        if corr_obj not in metrics_to_resolve and corr_obj[3] in ['METRIC','AGG_METRIC']: 
            metrics_to_resolve.append(corr_obj)
            print("### New metric found\n")
        # add newly found objects to the list
        if corr_obj not in newly_found: 
            newly_found.append(corr_obj)
            print("### New object added\n")
    return metrics_to_resolve, newly_found

def resolve_metrics(metrics_components):
    #TODO - it returns prompts, filters, transformations as schema objects - they yet to be resolved
    metrics_to_resolve=[obj for obj in metrics_components if obj[3] in ['METRIC','AGG_METRIC']]
    newly_found=[]
    
    # I am using while loop here, any newly found component metric is added to the list to be analyzed as well
    while metrics_to_resolve:
        for m in metrics_to_resolve:
            if m[3] == 'METRIC':
                print("\nMETRIC",m[1],m[2],m[3],m[5])
                oc=get_metric_definition(m[4],m[5])
                metrics_to_resolve, newly_found = dig_metric(m, oc, metrics_to_resolve, newly_found)
            elif m[3] == 'AGG_METRIC':
                print("\nAGG_METRIC",m[1],m[2],m[4],m[5])
                oc=get_base_formula_definition(conn,m[4],m[5])
                metrics_to_resolve, newly_found = dig_metric(m, oc, metrics_to_resolve, newly_found)
                
            metrics_to_resolve.remove(m)
            print(f"Number of metrics that remain to be resolved: {len(metrics_to_resolve)}")
    return newly_found

In [ ]:
'''
# Testing the functions
print(get_metric_definition("4C051DB611D3E877C000B3B2D86C964F","da"))
print(get_base_formula_definition(conn, "4C05176211D3E877C000B3B2D86C964F","bf"))
'''

In [32]:
metrics_list = list_metrics(connection=conn, project_name=project_name)
lenmetriclist=len(metrics_list)
metrics_components=[]

# first, build a list of top level metrics
for i, metric in enumerate(metrics_list[:]):
    if int(metric.subtype) not in [1025,1026]: # for some reason subtotals (1026) are returned as metrics (?)
        print(f"{i}/{lenmetriclist} - {metric.name} - subtype: {metric.subtype}")
        single_metric_components = get_metric_definition(metric.id, metric.name)
        print(single_metric_components)
        metrics_components.extend(single_metric_components)

0/703 - 1st Quartile - subtype: 1024
[['METRIC', 'C46205754204673DCEEB3B845C5796EA', '1st Quartile', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue'], ['METRIC', 'C46205754204673DCEEB3B845C5796EA', '1st Quartile', 'ATTRIBUTE', '96ED3EC811D5B117C000E78A4CC5F24F', 'Day']]
1/703 - 25th Percentile - subtype: 1024
[['METRIC', 'C77F998D11D60B4BC0008F916B98494F', '25th Percentile', 'PROMPT_OBJECTS', 'C77F972011D60B4BC0008F916B98494F', '[Choose a base fact.]'], ['METRIC', 'C77F998D11D60B4BC0008F916B98494F', '25th Percentile', 'PROMPT_OBJECTS', 'C77F972911D60B4BC0008F916B98494F', '[Choose a sample set level.]']]
2/703 - 25th Percentile of Revenue Per Customer - subtype: 1024
[['METRIC', 'AEF4592511D60B7BC0008F916B98494F', '25th Percentile of Revenue Per Customer', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue'], ['METRIC', 'AEF4592511D60B7BC0008F916B98494F', '25th Percentile of Revenue Per Customer', 'ATTRIBUTE', '8D679D3C11D3E4981000E787EC6DE8A4', 'Customer']]
3/703 - 3rd Quartile

[['METRIC', '5E2B3D8611D66F4BC0006CBEB630224F', 'Average Daily Sales', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue'], ['METRIC', '5E2B3D8611D66F4BC0006CBEB630224F', 'Average Daily Sales', 'ATTRIBUTE', '96ED3EC811D5B117C000E78A4CC5F24F', 'Day']]
31/703 - Average Inventory - subtype: 1024
[['METRIC', '67F9316611D51F1610005B89EBC3E6A4', 'Average Inventory', 'FACT', '4E01932F11D4F6D4C0004F916B98494F', '[End on hand]'], ['METRIC', '67F9316611D51F1610005B89EBC3E6A4', 'Average Inventory', 'ATTRIBUTE', '8D679D4211D3E4981000E787EC6DE8A4', 'Item'], ['METRIC', '67F9316611D51F1610005B89EBC3E6A4', 'Average Inventory', 'FILTER', '67F9316011D51F1610005B89EBC3E6A4', '[Year = Current year in sample DB]']]
32/703 - Average Monthly Volume - subtype: 1024
[['METRIC', 'ECB5AAB84215AF1A580D55BBB0F01A1D', 'Average Monthly Volume', 'METRIC', '67E6F0BE4CC6933DB65B058940189844', '[Monthly Volume]']]
33/703 - Average Quantity per Quarter - subtype: 1024
[['METRIC', 'A971BA0011D51ED410005B89EBC3E6A4', 'A

[['METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', 'Avg Revenue per Order', 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue'], ['METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', 'Avg Revenue per Order', 'METRIC', '4C05175211D3E877C000B3B2D86C964F', '[Order Count]']]
62/703 - Avg Revenue per Order Forecast - subtype: 1024
[['METRIC', 'E25EC482456BDD8BC3EEB4A3F7F9410F', 'Avg Revenue per Order Forecast', 'METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', '[Avg Revenue per Order]'], ['METRIC', 'E25EC482456BDD8BC3EEB4A3F7F9410F', 'Avg Revenue per Order Forecast', 'METRIC', '26007AC44155BE0E7CB53E89A07AABEC', '[AvgOrder Forecast Factor]']]
63/703 - Avg Revenue per Order Status - subtype: 1024
[['METRIC', 'E1B79DF54D98DD310D0A1988C2D7E157', 'Avg Revenue per Order Status', 'METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', '[Avg Revenue per Order]'], ['METRIC', 'E1B79DF54D98DD310D0A1988C2D7E157', 'Avg Revenue per Order Status', 'METRIC', 'E25EC482456BDD8BC3EEB4A3F7F9410F', '[Avg Revenue per Order Foreca

[['METRIC', '1FB34B414FE712C0E58A2FACBD0CA6CE', '% Change to Revenue', 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue'], ['METRIC', '1FB34B414FE712C0E58A2FACBD0CA6CE', '% Change to Revenue', 'METRIC', '4C05199311D3E877C000B3B2D86C964F', "[Last Year's Revenue]"]]
89/703 - % Change to Units Sold - subtype: 1024
[['METRIC', 'AD0C38604223BD91C3CED3B7F78C0CAB', '% Change to Units Sold', 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue'], ['METRIC', 'AD0C38604223BD91C3CED3B7F78C0CAB', '% Change to Units Sold', 'METRIC', '4C05199311D3E877C000B3B2D86C964F', "[Last Year's Revenue]"]]
90/703 - % Change YTD Sales - subtype: 1024
[['METRIC', 'B0971D1F43BAFE65137FDF819EECFB6B', '% Change YTD Sales', 'METRIC', 'FA31599341111F2BF9918E8704FA17DC', 'YTDRevenue'], ['METRIC', 'B0971D1F43BAFE65137FDF819EECFB6B', '% Change YTD Sales', 'METRIC', '84A4D59D4B964E48D82DF18E0E76413C', '[LY-YTDRevenue]']]
91/703 - % Change YTD Units Sold - subtype: 1024
[['METRIC', 'C73A76B04070FEE11A15CB8071456700

[['METRIC', 'FBE5F7FD4A7A2CA358DF5DBC77DD9BF8', 'Cost/Unit Sold', 'METRIC', '7FD5B69611D5AC76C000D98A4CC5F24F', 'Cost'], ['METRIC', 'FBE5F7FD4A7A2CA358DF5DBC77DD9BF8', 'Cost/Unit Sold', 'METRIC', '4C05190A11D3E877C000B3B2D86C964F', '[Units Sold]']]
113/703 - Cost/Unit Sold Forecast - subtype: 1024
[['METRIC', 'D9E22B8947C8F6E4B177C58BA3B4D4B3', 'Cost/Unit Sold Forecast', 'METRIC', 'FBE5F7FD4A7A2CA358DF5DBC77DD9BF8', '[Cost/Unit Sold]']]
115/703 - Count Distinct (Items Sold) - subtype: 1024
[['METRIC', 'A73DF7944B47BA98D23110A63FC17C90', 'Count Distinct (Items Sold)', 'FACT', '62AA362011D3E4EC1000E887EC6DE8A4', '[Item-level Units Sold]'], ['METRIC', 'A73DF7944B47BA98D23110A63FC17C90', 'Count Distinct (Items Sold)', 'ATTRIBUTE', '8D679D4211D3E4981000E787EC6DE8A4', 'Item']]
116/703 - Count of Customers - subtype: 1024
[['METRIC', 'AEF4590D11D60B7BC0008F916B98494F', 'Count of Customers', 'ATTRIBUTE', '8D679D3C11D3E4981000E787EC6DE8A4', 'Customer']]
117/703 - Count of Customers - subtype: 1

...

[['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', '09ADC89546E03A593AA73F88C03C11BF', 'AvgMinOffPeak'], ['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', 'FCF84C4F41206BDC731E0995EDA98502', 'AvgMinPeak'], ['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', 'C70876154C180F8C769194A86E6562C7', 'DroppedCalls'], ['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', '707B3E304D52877B93F9C8B15EDBECEB', 'HelpdeskCalls'], ['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', '7015E607441BB8501EE4409DB671432D', '[Age Range]'], ['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', '7456B3DC409DFE56354CE7AA0CE1CEC5', 'Gender'], ['METRIC_DMX', 'D0D9053243479449FD9591B32C8B69E4', 'Telco Churn Propensity (Tree)', 'METRIC', '313550BE4CDF0413EB97C1B117B08

[['METRIC_TRAINING', '0E2607B543C57BC068B7B39D8D3E2A06', "Training metric for 'Customer Segment'", 'METRIC', '7015E607441BB8501EE4409DB671432D', '[Age Range]'], ['METRIC_TRAINING', '0E2607B543C57BC068B7B39D8D3E2A06', "Training metric for 'Customer Segment'", 'METRIC', 'A2F1900048CB41F80AAB1AAA377EC860', 'Education'], ['METRIC_TRAINING', '0E2607B543C57BC068B7B39D8D3E2A06', "Training metric for 'Customer Segment'", 'METRIC', '7456B3DC409DFE56354CE7AA0CE1CEC5', 'Gender'], ['METRIC_TRAINING', '0E2607B543C57BC068B7B39D8D3E2A06', "Training metric for 'Customer Segment'", 'METRIC', '69DFE8FA4DBBEF9F684EFE8D08AD6924', '[Housing Type]'], ['METRIC_TRAINING', '0E2607B543C57BC068B7B39D8D3E2A06', "Training metric for 'Customer Segment'", 'METRIC', '6EE95F024F729E8FF2364F96B5DAC7B2', '[Marital Status]']]
628/703 - Training metric for 'Customer Segment (Optimal)' - subtype: 1028
[['METRIC_TRAINING', '6499D6E8415F622CA844DF804246E6D4', "Training metric for 'Customer Segment (Optimal)'", 'METRIC', '701

[['METRIC_TRAINING', 'DD0CDD9F4A9FBC567C24A286C19254A2', "Training metric for 'Revenue Predictor (Daily by Call Center)'", 'METRIC', 'CEE087824E0CDE1FFE9E8693108FB91F', '[Call Center]'], ['METRIC_TRAINING', 'DD0CDD9F4A9FBC567C24A286C19254A2', "Training metric for 'Revenue Predictor (Daily by Call Center)'", 'METRIC', '98E1D8644AA6B6010F5FD9BB40BC938A', '[Day Index]'], ['METRIC_TRAINING', 'DD0CDD9F4A9FBC567C24A286C19254A2', "Training metric for 'Revenue Predictor (Daily by Call Center)'", 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue']]
640/703 - Training metric for 'Revenue Predictor (Exponential)' - subtype: 1028
[['METRIC_TRAINING', 'FA0ADFE74E2B1E573A3DF1976EC41FDD', "Training metric for 'Revenue Predictor (Exponential)'", 'METRIC', 'C35C9E204AA26758B3C630AA3CC54D34', '[Quarter Index]'], ['METRIC_TRAINING', 'FA0ADFE74E2B1E573A3DF1976EC41FDD', "Training metric for 'Revenue Predictor (Exponential)'", 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue']]
641/703 - Training

[['METRIC_TRAINING', '35CB2D5F4660FADE61E7709B565C5911', "Training Metric for 'Web Order Ratio Predictor'", 'METRIC', '547B7F734881524948CC2196CDE3C1DA', '[Month Index]'], ['METRIC_TRAINING', '35CB2D5F4660FADE61E7709B565C5911', "Training Metric for 'Web Order Ratio Predictor'", 'METRIC', '1EF275104853682DC1DDE8B58AFC1B12', '[Web Order Ratio]']]
652/703 - Training Metric for 'Web Order Ratio Predictor (Monthly Seasonality)' - subtype: 1028
[['METRIC_TRAINING', '01E5F49A4873BA280DF5328D546757BD', "Training Metric for 'Web Order Ratio Predictor (Monthly Seasonality)'", 'METRIC', '547B7F734881524948CC2196CDE3C1DA', '[Month Index]'], ['METRIC_TRAINING', '01E5F49A4873BA280DF5328D546757BD', "Training Metric for 'Web Order Ratio Predictor (Monthly Seasonality)'", 'METRIC', '1EF275104853682DC1DDE8B58AFC1B12', '[Web Order Ratio]']]
653/703 - Train SPSS - subtype: 1024
[['METRIC', '0EBDA70C4EDE7E1000F3B4AFA5B4B904', 'Train SPSS', 'ATTRIBUTE', '8D679D5111D3E4981000E787EC6DE8A4', 'Year']]
654/703 -

[['METRIC', '9E31DECA494701787E544485C2B7462C', 'Units Sold Rand Metric', 'METRIC', '4C05190A11D3E877C000B3B2D86C964F', '[Units Sold]']]
679/703 - Units Sold Status - subtype: 1024
[['METRIC', '803BE0704085BDE65FEA55B03E83D43B', 'Units Sold Status', 'METRIC', '4C05190A11D3E877C000B3B2D86C964F', '[Units Sold]'], ['METRIC', '803BE0704085BDE65FEA55B03E83D43B', 'Units Sold Status', 'METRIC', '153C9DFF4CDB947357F57D9E80452F38', '[Units Sold Forecast]']]
680/703 - Units Sold Target - subtype: 1024
[['METRIC', '1795223F4B806809F95544A32C501DF5', 'Units Sold Target', 'FACT', 'F44AA4E34CADBE24A5084AB550B5FD6A', '[Units Sold Target]']]
681/703 - Units Sold vs. Target - subtype: 1024
[['METRIC', 'D2CF060F44636FA55AD3EEBA7837218B', 'Units Sold vs. Target', 'METRIC', '4C05190A11D3E877C000B3B2D86C964F', '[Units Sold]']]
682/703 - Units Sold - YTD - subtype: 1024
[['METRIC', '2FE0335545064BEDA5CC429CB4DB9691', 'Units Sold - YTD', 'FACT', '8D67958511D3E4981000E787EC6DE8A4', '[Units Sold]'], ['METRIC',

In [33]:
%%time
# metrics_components is a list of inner metrics - metrics that top level metrics are based on
# we will analyze them recursively until we dig in to the facts - we stop there
# for the main visualization only metric-fact relations will be displayed
# however, metric-metric relations will be added to the dataset for anyone interested in checking it

nf=resolve_metrics(metrics_components)


METRIC 707CC3364810929BE8617FAFE7114639 A METRIC [DeltaLat/2]
['METRIC', 'CE0B238C4D0E3B07CFAEEDA17CA08DA4', '[DeltaLat/2]', 'METRIC', 'C14E9B0B47B436889A894297B5C826D9', 'Store_Lat']
### New object added

['METRIC', 'CE0B238C4D0E3B07CFAEEDA17CA08DA4', '[DeltaLat/2]', 'PROMPT_DOUBLE', 'D68E8D524CF6B8F56A90A5AB7FEE055A', 'Latitude']
### New object added

Number of metrics that remain to be resolved: 870

METRIC 707CC3364810929BE8617FAFE7114639 A METRIC [DeltaLong/2]
['METRIC', '1412509C47422B90BE15C2B201E1CD2D', '[DeltaLong/2]', 'METRIC', 'E6F137D142C2B9B565FC5E93C37709B1', 'Store_Long']
### New metric found

### New object added

['METRIC', '1412509C47422B90BE15C2B201E1CD2D', '[DeltaLong/2]', 'PROMPT_DOUBLE', '0967118A402ACF2F1A4DC0AC10D0182D', 'Longitude']
### New object added

Number of metrics that remain to be resolved: 870

METRIC 301B7B814449FB4E97B2C4B1EC0C5F6B Affinity METRIC [Reference Item Customer Count]
['METRIC', '69E6DB3211D3E8B3C000B3B2D86C964F', '[Reference Item Custom

['METRIC', '579FDA654404D303C2992688EF200073', '[Avg Revenue per Customer]', 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

['METRIC', '579FDA654404D303C2992688EF200073', '[Avg Revenue per Customer]', 'METRIC', '4C05174511D3E877C000B3B2D86C964F', '[Customer Count]']
### New metric found

### New object added

Number of metrics that remain to be resolved: 871

METRIC EB17F0554D93FDA87A6BA3A48BFA9074 Avg Revenue per Order METRIC Revenue
['METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

Number of metrics that remain to be resolved: 871

METRIC E25EC482456BDD8BC3EEB4A3F7F9410F Avg Revenue per Order Forecast METRIC [Avg Revenue per Order]
['METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', '[Avg Revenue per Order]', 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

['METRIC', 'EB17

['METRIC', '4C051E2F11D3E877C000B3B2D86C964F', '[Combined Revenue]', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

['METRIC', '4C051E2F11D3E877C000B3B2D86C964F', '[Combined Revenue]', 'FILTER', '69E6DA3611D3E8B3C000B3B2D86C964F', '[Customers who bought Reference Item]']
### New object added

Number of metrics that remain to be resolved: 860

METRIC 3D7675A84EF89DF3C3CA3E8BC2A3180A Contribution % to Yearly Profit METRIC Profit
['METRIC', '4C051DB611D3E877C000B3B2D86C964F', 'Profit', 'FACT', 'F30CB66411D5AC5FC000D98A4CC5F24F', 'Profit']
### New object added

Number of metrics that remain to be resolved: 859

METRIC DE66B0144D4E2D0042A61FB576A5B04F Contribution % to Yearly Revenue METRIC Revenue
['METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

Number of metrics that remain to be resolved: 859

METRIC 69E6DC2A11D3E8B3C0

['METRIC', '69E6DB9A11D3E8B3C000B3B2D86C964F', '[Total Customer Count]', 'ATTRIBUTE', '8D679D3C11D3E4981000E787EC6DE8A4', 'Customer']
### New object added

['METRIC', '69E6DB9A11D3E8B3C000B3B2D86C964F', '[Total Customer Count]', 'ATTRIBUTE', '8D679D4211D3E4981000E787EC6DE8A4', 'Item']
### New object added

Number of metrics that remain to be resolved: 853

METRIC 5ED35C6F48300E805354D88FB3067D53 Customer Segment METRIC Education
['METRIC', 'A2F1900048CB41F80AAB1AAA377EC860', 'Education', 'ATTRIBUTE', 'D40E5ACA409A88F64BDE4FB2B29646E9', '[Education Level]@DESC']
### New object added

['METRIC', 'A2F1900048CB41F80AAB1AAA377EC860', 'Education', 'ATTRIBUTE', '8D679D3C11D3E4981000E787EC6DE8A4', 'Customer']
### New object added

Number of metrics that remain to be resolved: 852

METRIC 5ED35C6F48300E805354D88FB3067D53 Customer Segment METRIC [Housing Type]
['METRIC', '69DFE8FA4DBBEF9F684EFE8D08AD6924', '[Housing Type]', 'ATTRIBUTE', '606595154DEEF0EBFB2BAA984562998E', '[Housing Type]@DESC']


['METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', '[Avg Revenue per Order]', 'METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

['METRIC', 'EB17F0554D93FDA87A6BA3A48BFA9074', '[Avg Revenue per Order]', 'METRIC', '4C05175211D3E877C000B3B2D86C964F', '[Order Count]']
### New metric found

### New object added

Number of metrics that remain to be resolved: 847

METRIC E6C9DE9F47A3253B4294D895E1E22797 % Diff from TM LY - Customer Count METRIC [Customer Count]
['METRIC', '4C05174511D3E877C000B3B2D86C964F', '[Customer Count]', 'AGG_METRIC', '4C05170411D3E877C000B3B2D86C964F', '[Customer Count from Fact]']
### New metric found

### New object added

Number of metrics that remain to be resolved: 847

METRIC 08C0B94F4289C52592B40287EAB18E68 % Diff from TM LY - Order Count METRIC [Order Count]
['METRIC', '4C05175211D3E877C000B3B2D86C964F', '[Order Count]', 'AGG_METRIC', '4C05171E11D3E877C000B3B2D86C964F', '[Order Count from Fact]']
### New metric fou

...

['METRIC', '4C05190A11D3E877C000B3B2D86C964F', '[Units Sold]', 'FACT', '8D67958511D3E4981000E787EC6DE8A4', '[Units Sold]']
Number of metrics that remain to be resolved: 64

AGG_METRIC 8AEEF1E04627E3BE78CB6A82F9A1D0F1 Trend - % Diff from LM - Avg per Order 4C05176211D3E877C000B3B2D86C964F Revenue
['AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
Number of metrics that remain to be resolved: 63

AGG_METRIC 3E7650AF4045FC6A88849F9A62263EEF Revenue % of Target 4C05176211D3E877C000B3B2D86C964F Revenue
['AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
Number of metrics that remain to be resolved: 62

AGG_METRIC 9D61FC824D5A68A2D40C60A2E9C9C523 Revenue Forecast Factor 4C05176211D3E877C000B3B2D86C964F Revenue
['AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
Number of metrics that remain to be re

['METRIC', '2744CCCA42722584559B289D27433DFD', '[AvgCust Rand Metric]', 'METRIC', '579FDA654404D303C2992688EF200073', '[Avg Revenue per Customer]']
Number of metrics that remain to be resolved: 44

METRIC FD9C9A1749C8332F24B50FA48F96AE82 % Diff from LM - Avg per Customer METRIC [Last Month's Revenue]
['METRIC', '4C05198411D3E877C000B3B2D86C964F', "[Last Month's Revenue]", 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

['METRIC', '4C05198411D3E877C000B3B2D86C964F', "[Last Month's Revenue]", 'ROLE_TRANSFORMATION', '6CB9ABDC11D3E4F11000E887EC6DE8A4', "[Last Month's]"]
Number of metrics that remain to be resolved: 44

METRIC 8E9CF3F54D810AB7F90034AB2DBF6B79 Training metric for 'Revenue Predictor (Monthly by Call Center)' METRIC [Month of Year]
['METRIC', '50EA23F94953A40F355488AB6F1B2AED', '[Month of Year]', 'FACT', 'FEDCF3A5404E070D3CBDA2A1DE70A804', 'MonthOfYear']
['METRIC', '50EA23F94953A40F355488AB6F1B2AED', '[Month of Year]', 'ATTRIBUTE', '8D679D441

['AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
Number of metrics that remain to be resolved: 29

METRIC 8D7D59C144AB891C57FEFA873ABE1C92 Revenue Forecast METRIC Revenue
['METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

Number of metrics that remain to be resolved: 29

METRIC 6C9E5797413BF3EE176D0A8DB9C703F8 Margin Forecast METRIC Revenue
['METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

Number of metrics that remain to be resolved: 29

METRIC F8CF107447EC3120A7539489229B2D1C Max. Avgorder METRIC Revenue
['METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

### New object added

Number of metrics that remain to be resolved: 29

METRIC 6B4F34E44DB31318F76

['AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
Number of metrics that remain to be resolved: 18

METRIC 8C8191034AA7862737DE12971936AD46 % Delta Profit Vs Forecast METRIC Profit
['METRIC', '4C051DB611D3E877C000B3B2D86C964F', 'Profit', 'FACT', 'F30CB66411D5AC5FC000D98A4CC5F24F', 'Profit']
Number of metrics that remain to be resolved: 17

METRIC 6B4F34E44DB31318F76CB380A6572BAB Avg Revenue per Customer Forecast METRIC Revenue
['METRIC', '4C05177011D3E877C000B3B2D86C964F', 'Revenue', 'AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue']
### New metric found

Number of metrics that remain to be resolved: 17

METRIC 8E1B4B1D45FD49A64804A18F405AEDC4 % Delta Cost Vs Forecast METRIC [Profit Forecast Percentage]
['METRIC', '6A802EE542E588DA012318B559BD86A9', '[Profit Forecast Percentage]', 'METRIC', '4C051DB611D3E877C000B3B2D86C964F', 'Profit']
Number of metrics that remain to be resolved: 16

METRIC 509A24BD42275CAEF4

['AGG_METRIC', '4C05176211D3E877C000B3B2D86C964F', 'Revenue', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
Number of metrics that remain to be resolved: 0
CPU times: total: 5.8 s
Wall time: 4min 46s


In [34]:
# update the main list with found dependecies
print("Number of top level metrics ",len(metrics_components))
print("Number of newly found dependent objects", len(nf))

Number of top level metrics  1443
Number of newly found dependent objects 1743


In [35]:
# add only unique new objects
# again, metric-metric relations are added, but will not be displayed on the main visualization
for n in nf:
    if n not in metrics_components:
        print(n)
        metrics_components.append(n)
        
#         
print("\nUpdated list of report objects (count) ", len(metrics_components))

['METRIC', '707CC3364810929BE8617FAFE7114639', 'A', 'METRIC', 'E6F137D142C2B9B565FC5E93C37709B1', 'Store_Long']
['METRIC', '707CC3364810929BE8617FAFE7114639', 'A', 'PROMPT_DOUBLE', '0967118A402ACF2F1A4DC0AC10D0182D', 'Longitude']
['METRIC', '301B7B814449FB4E97B2C4B1EC0C5F6B', 'Affinity', 'AGG_METRIC', '4C05170411D3E877C000B3B2D86C964F', '[Customer Count from Fact]']
['METRIC', '301B7B814449FB4E97B2C4B1EC0C5F6B', 'Affinity', 'ATTRIBUTE', '8D679D4211D3E4981000E787EC6DE8A4', 'Item']
['METRIC', '301B7B814449FB4E97B2C4B1EC0C5F6B', 'Affinity', 'FILTER', '69E6D9DD11D3E8B3C000B3B2D86C964F', '[Reference Item]']
['METRIC', '4C051B8D11D3E877C000B3B2D86C964F', 'All Customers Revenue Abs.', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
['METRIC', '4C051B7E11D3E877C000B3B2D86C964F', 'All Products Revenue Abs.', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
['METRIC', 'CD40A36844E25F955C55E3A453A76002', 'All Regions (Abs)', 'FACT', '8D67958411D3E4981000E787EC6DE8A4', 'Revenue']
['ME

### Add attributes to the list

In [36]:
# Metrics and attributes are the most interesting report objects to be analyzed
report_objects=metrics_components+list_attr_rep_obj
print(len(report_objects))

3214


### Export CSV

In [37]:
headers_datasets=["repobj_type", "repobj_id", "repobj_name", "schemaobj_type" , "schemaobj_id", "schemaobj_name"]
df = pd.DataFrame(report_objects, columns = headers_datasets)
df.to_csv(f"L3-Report_objects.csv", index=False, encoding='utf-8', sep=",", 
          quoting=csv.QUOTE_ALL, escapechar="\\", header=True)

# Scratchpad

In [ ]:
attr.list_properties()
vars(attr)
print(r.__dict__)

attr = Attribute(connection=conn, id='8D679D3711D3E4981000E787EC6DE8A4')

### Test recursion

In [ ]:
aaa=[1,2,3,4,5,6]
while aaa:
    for a in aaa:
        print(">>> ",a)
        if a in [5,6]:
            aaa.append(a+10)
            print("added ",a+10)        
        aaa.remove(a)
        print(f"removed {a}")

### Fact

In [45]:
from mstrio.object_management import full_search
from mstrio.types import ObjectSubTypes, ObjectTypes

PROJECT_ID = "B7CA92F04B9FAE8D941C3E9B7E0CD754"
FACT_ID = "3103680A41C81E81EC3079BD44126F09"

objects = full_search(
    conn,
    project=PROJECT_ID,
    uses_object_id=FACT_ID,
    uses_object_type=ObjectTypes.FACT,
    object_types=[ObjectTypes.METRIC, ObjectTypes.AGG_METRIC],
    uses_recursive=True
)
print("Number of dependent metrics: ",len(objects))

Number of dependent metrics:  635


In [43]:
objects

[{'name': '1st Quartile',
  'id': 'C46205754204673DCEEB3B845C5796EA',
  'type': 4,
  'subtype': 1024,
  'date_created': '2005-12-01T21:47:40.000+0000',
  'date_modified': '2016-08-12T19:33:22.000+0000',
  'version': 'CFFFA103458375AE2E48408C206875D2',
  'acg': 255,
  'owner': {'name': 'Administrator', 'id': '54F3D26011D2896560009A8E67019608'},
  'acl': [{'deny': False,
    'type': 1,
    'rights': 199,
    'trusteeId': '294DEDC011D2F1D56000D98E67019608',
    'trusteeName': 'Public / Guest',
    'trusteeType': 34,
    'trusteeSubtype': 8705,
    'inheritable': False},
   {'deny': False,
    'type': 1,
    'rights': 199,
    'trusteeId': 'C82C6B1011D2894CC0009D9F29718E4F',
    'trusteeName': 'Everyone',
    'trusteeType': 34,
    'trusteeSubtype': 8705,
    'inheritable': False},
   {'deny': False,
    'type': 1,
    'rights': 255,
    'trusteeId': '54F3D26011D2896560009A8E67019608',
    'trusteeName': 'Administrator',
    'trusteeType': 34,
    'trusteeSubtype': 8704,
    'inheritable':

In [48]:
import mstrio
print(mstrio.__version__)

11.3.7.103
